# w9_i2ce_continue.ipynb — 固定分割单塔 i2ce@4096 续训 2000 -> 4000ep

User: 只对之前单次跑到 2000ep 的那一个 checkpoint 继续训练，要事实不要
猜想。Target = `w9_wcle_i2ce_icetf_g4096_fp`（scale pod 已于 07-16 将其
续至 ep2000；ckpt_ep2000.pt 与 tower_ep2000.npz 均已在桶上确认）。fs
worker 自带 EXTEND-from-ckpt：取最新 ckpt 权重、fresh opt/amp/rng，训练
2000->4000；ZS sweep 只评新增 ckpt（旧投影 npz 秒跳），zsbest 在完整
50..4000 曲线上按 zvsel 重新选点。单塔单卡（grad gallery ~45G，A100-80G），
约 +20h。ARMS 常量可加 "wcle_ce_cetf" 让配对基线跟进。NOT the CV folds —
final_experiment 的五折塔不受影响。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # fixed-split campaign dir

ARMS = ["wcle_i2ce_icetf"]             # add "wcle_ce_cetf" for the paired ref
CAP = 4096
EPOCHS_BASE = 2000                     # extend only from this budget
EPOCHS = 4000
os.makedirs(OUT_DIR, exist_ok=True)
print("targets:", [f"w9_{a}_g{CAP}" for a in ARMS], f"-> {EPOCHS}ep")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# EXTEND run: one tower per GPU, sequential over ARMS (usually just one).
# Gate: ep{BASE} npz present (first leg done) AND ep{EPOCHS} npz absent.
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
todo = []
for arm in ARMS:
    nm = J.fs_label(arm, CAP, False, 0, "clean", 16)
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} already at {EPOCHS}"); continue
    if not (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS_BASE}.npz").exists():
        print(f"[wait] {nm} not at {EPOCHS_BASE} yet -- run scale first"); continue
    todo.append((arm, nm))
print(f"{len(todo)} tower(s) to extend")

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, arm, nm):
    if not J.try_claim(cdir, nm):
        # corpse window: a pod that died <120s ago still looks alive.
        # Wait out DEAD_SEC once and retry before giving up (fast relaunch
        # otherwise skips everything and auto-stops -- looks like a crash).
        print(f"[claim] {nm} fresh/held -- waiting 130s for the corpse "
              "window, then retrying once", flush=True)
        time.sleep(130)
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
           "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
           "--topup-seeds", str(J.TOPUP_SEEDS),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] EXTEND {nm} -> {EPOCHS}ep", flush=True)
    t0 = time.time()
    with open(logd / f"{arm}_g{CAP}_ext{EPOCHS}.log", "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
    if p.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

ths = [threading.Thread(target=run_one, args=(g, arm, nm))
       for g, (arm, nm) in zip(gpus, todo)]
for t in ths:
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# FACTS: best<=2000 vs best<=4000 (zvsel-selected from the traj) + tail
# windows. No interpretation.
import json
import numpy as np
from pathlib import Path

def wmean(tr, eps, key, lo, hi):
    v = [tr[e][key] for e in eps if lo <= int(e[2:]) <= hi and key in tr[e]]
    return float(np.mean(v)) if v else float("nan")

for arm in ARMS:
    nm = J.fs_label(arm, CAP, False, 0, "clean", 16)
    p = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    if not p.exists():
        print(f"{nm}: (no traj)"); continue
    tr = json.loads(p.read_text())
    eps = sorted(tr, key=lambda e: int(e[2:]))
    cand = [e for e in eps if "zvsel" in tr[e]]
    def best(upto):
        cs = [e for e in cand if int(e[2:]) <= upto]
        return max(cs, key=lambda e: (tr[e]["zvsel"], -int(e[2:]))) if cs else None
    b1, b2 = best(EPOCHS_BASE), best(10**9)
    print(f"===== {nm} (traj to ep{eps[-1][2:]}) =====")
    for tag_, b in (("best<=2000", b1), ("best<=4000", b2)):
        if not b:
            print(f"  {tag_}: (none)"); continue
        d = tr[b]
        print(f"  {tag_}: ep{b[2:]:>4}  neu {d['nm_neutral']:.3f} "
              f"non {d['nm_noname']:.3f} h5non {d.get('h5_noname', float('nan')):.3f} "
              f"tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f} zvsel {d['zvsel']:.3f}")
    for key in ("nm_neutral", "nm_noname", "tag_noname", "zvsel"):
        a = wmean(tr, eps, key, 1500, 2000)
        b = wmean(tr, eps, key, 3500, 4000)
        print(f"  {key:10s} win[1500-2000] {a:.3f} -> win[3500-4000] {b:.3f} ({b - a:+.3f})")


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)